This implementation performs a fully scalable, generic systolic array architecture that can be compiled to any size.
The key to this flexibility is the use of genvar syntax combined with nested generate loops in the structural RTL (sysarray.v). Instead of manually copying, pasting, and hard-coding processing elements, the compiler calculates the grid layout dynamically during elaboration.

How to Change the Array Size:

1. Modify the RTL Parameters: By simply changing the values of the ROW and COL parameters inside your top-level module instantiation, the hardware description compiler will instantly generate the appropriate array grid size (e.g., resizing from 4 × 4 to 8 × 8 or 16 × 16).
2. Update the Testbench: When changing the structural size of the array, the testbench must be updated to match the chosen ROW and COL bounds. This ensures that:
	• The matrix ingestion timelines are properly skewed (Total Steps = 2 × (ROW - 1) + Length).
	• The physical data injection wires map correctly to the updated external boundaries of the matrix grid.


In [17]:
!apt-get install iverilog


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (12.0-2build2).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [18]:
%%writefile PE.v
module PE #(
    parameter DATA_WIDTH = 16,
    parameter ACC_WIDTH  = 32
)(
    input wire clk, rst,
    input wire signed [DATA_WIDTH-1:0] a_in, b_in,
    output reg signed [DATA_WIDTH-1:0] a_out, b_out,
    output reg signed [ACC_WIDTH-1:0]  c_out
);

always @(posedge clk or posedge rst)
begin
    if (rst == 1'b1)
    begin
        a_out <= 0;
        b_out <= 0;
        c_out <= 0;
    end
    else
    begin
        a_out <= a_in;
        b_out <= b_in;
        c_out <= c_out + (a_in * b_in);
    end
end

endmodule


Overwriting PE.v


In [19]:
%%writefile sysarray.v
module SystolicArrayMod #(
    parameter DATA_WIDTH = 16,
    parameter ACC_WIDTH  = 32,
    parameter ROW = 4,
    parameter COL = 4
)(
    input wire Clk, Rst,
    input wire signed [DATA_WIDTH-1:0] A_row0, A_row1, A_row2, A_row3,
    input wire signed [DATA_WIDTH-1:0] B_col0, B_col1, B_col2, B_col3,
    output wire signed [ACC_WIDTH-1:0] C_00, C_01, C_02, C_03, C_10, C_11, C_12, C_13, C_20, C_21, C_22, C_23, C_30, C_31, C_32, C_33
);

    // Internal wire arrays for interconnects
    wire signed [DATA_WIDTH-1:0] A [0:ROW-1][0:COL];
    wire signed [DATA_WIDTH-1:0] B [0:ROW][0:COL-1];
    wire signed [ACC_WIDTH-1:0]  C [0:ROW-1][0:COL-1];

    // Explicitly mapping Row Inputs to Row Index 0 of the internal A array
    assign A[0][0] = A_row0;
    assign A[1][0] = A_row1;
    assign A[2][0] = A_row2;
    assign A[3][0] = A_row3;

    // Explicitly mapping Column Inputs to Column Index 0 of the internal B array
    assign B[0][0] = B_col0;
    assign B[0][1] = B_col1;
    assign B[0][2] = B_col2;
    assign B[0][3] = B_col3;

    // Mapping internal Accumulator results out to top-level individual port wires
    assign C_00 = C[0][0]; assign C_01 = C[0][1]; assign C_02 = C[0][2]; assign C_03 = C[0][3];
    assign C_10 = C[1][0]; assign C_11 = C[1][1]; assign C_12 = C[1][2]; assign C_13 = C[1][3];
    assign C_20 = C[2][0]; assign C_21 = C[2][1]; assign C_22 = C[2][2]; assign C_23 = C[2][3];
    assign C_30 = C[3][0]; assign C_31 = C[3][1]; assign C_32 = C[3][2]; assign C_33 = C[3][3];

    // Generate loop for the 4x4 matrix processing elements
    genvar r, c;
    generate
        for (r=0; r<ROW; r=r+1) begin: genrow
            for (c=0; c<COL; c=c+1) begin: gencol
                PE PE_inst (
                    .clk   (Clk),
                    .rst   (Rst),
                    .a_in  (A[r][c]),
                    .b_in  (B[r][c]),
                    .a_out (A[r][c+1]),
                    .b_out (B[r+1][c]),
                    .c_out (C[r][c])
                );
            end
        end
    endgenerate

endmodule


Overwriting sysarray.v


In [20]:
%%writefile tb.v
module testbench;

    parameter DATA_WIDTH = 16;
    parameter ACC_WIDTH  = 32;
    parameter ROW        = 4;
    parameter COL        = 4;
    parameter Length     = 4;

    reg clk_test;
    reg rst_test;

    reg signed [DATA_WIDTH-1:0] A_row_test [0:ROW-1];
    reg signed [DATA_WIDTH-1:0] B_col_test [0:COL-1];

    wire signed [ACC_WIDTH-1:0] C_out_00, C_out_01, C_out_02, C_out_03;
    wire signed [ACC_WIDTH-1:0] C_out_10, C_out_11, C_out_12, C_out_13;
    wire signed [ACC_WIDTH-1:0] C_out_20, C_out_21, C_out_22, C_out_23;
    wire signed [ACC_WIDTH-1:0] C_out_30, C_out_31, C_out_32, C_out_33;

    reg signed [DATA_WIDTH-1:0] A [0:ROW-1][0:Length-1];
    reg signed [DATA_WIDTH-1:0] B [0:Length-1][0:COL-1];

    integer t, r, c;
    integer col_idx, row_idx;

    SystolicArrayMod #(
        .DATA_WIDTH(DATA_WIDTH),
        .ACC_WIDTH(ACC_WIDTH),
        .ROW(ROW),
        .COL(COL)
    ) dut (
        .Clk(clk_test),
        .Rst(rst_test),
        .A_row0(A_row_test[0]), .A_row1(A_row_test[1]), .A_row2(A_row_test[2]), .A_row3(A_row_test[3]),
        .B_col0(B_col_test[0]), .B_col1(B_col_test[1]), .B_col2(B_col_test[2]), .B_col3(B_col_test[3]),
        .C_00(C_out_00), .C_01(C_out_01), .C_02(C_out_02), .C_03(C_out_03),
        .C_10(C_out_10), .C_11(C_out_11), .C_12(C_out_12), .C_13(C_out_13),
        .C_20(C_out_20), .C_21(C_out_21), .C_22(C_out_22), .C_23(C_out_23),
        .C_30(C_out_30), .C_31(C_out_31), .C_32(C_out_32), .C_33(C_out_33)
    );

    always #10 clk_test = ~clk_test;

    initial begin
        clk_test = 0;
        rst_test = 1;

        for (r = 0; r < ROW; r = r + 1) begin
            A_row_test[r] = {DATA_WIDTH{1'b0}};
        end
        for (c = 0; c < COL; c = c + 1) begin
            B_col_test[c] = {DATA_WIDTH{1'b0}};
        end

        // Generate known input matrices
        for (r = 0; r < ROW; r = r + 1) begin
            for (c = 0; c < Length; c = c + 1) begin
                A[r][c] = r + c + 1;
                B[c][r] = c + r + 2;
            end
        end

        // Print input matrices first
        $display("\n=== INPUT MATRIX A ===");
        for (r = 0; r < ROW; r = r + 1) begin
            $display("[%3d, %3d, %3d, %3d]", A[r][0], A[r][1], A[r][2], A[r][3]);
        end

        $display("\n=== INPUT MATRIX B ===");
        for (r = 0; r < Length; r = r + 1) begin
            $display("[%3d, %3d, %3d, %3d]", B[r][0], B[r][1], B[r][2], B[r][3]);
        end

        #40;
        rst_test = 0; // Release Reset

        // Feeding data in step-by-step with a skew factor
        for (t = 0; t < ((2 * (ROW - 1)) + Length); t = t + 1) begin
            @ (negedge clk_test);

            for (r = 0; r < ROW; r = r + 1) begin
                col_idx = t - r;
                if (col_idx >= 0 && col_idx < Length) begin
                    A_row_test[r] = A[r][col_idx];
                end else begin
                    A_row_test[r] = {DATA_WIDTH{1'b0}};
                end
            end

            for (c = 0; c < COL; c = c + 1) begin
                row_idx = t - c;
                if (row_idx >= 0 && row_idx < Length) begin
                    B_col_test[c] = B[row_idx][c];
                end else begin
                    B_col_test[c] = {DATA_WIDTH{1'b0}};
                end
            end
        end

        // Wait for last element computation to settle inside PE arrays
        @(negedge clk_test);
        #20;

        $display("\n=== SIMULATION COMPLETE! OUTPUT MATRIX C ===");
        $display("[%5d, %5d, %5d, %5d]", C_out_00, C_out_01, C_out_02, C_out_03);
        $display("[%5d, %5d, %5d, %5d]", C_out_10, C_out_11, C_out_12, C_out_13);
        $display("[%5d, %5d, %5d, %5d]", C_out_20, C_out_21, C_out_22, C_out_23);
        $display("[%5d, %5d, %5d, %5d]", C_out_30, C_out_31, C_out_32, C_out_33);
        $display("===========================================\n");
        $finish;
    end

endmodule


Overwriting tb.v


In [21]:
!apt-get install -y iverilog



Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (12.0-2build2).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [22]:
!iverilog -o systolic_sim PE.v sysarray.v tb.v


In [23]:
!vvp systolic_sim



=== INPUT MATRIX A ===
[  1,   2,   3,   4]
[  2,   3,   4,   5]
[  3,   4,   5,   6]
[  4,   5,   6,   7]

=== INPUT MATRIX B ===
[  2,   3,   4,   5]
[  3,   4,   5,   6]
[  4,   5,   6,   7]
[  5,   6,   7,   8]

=== SIMULATION COMPLETE! OUTPUT MATRIX C ===
[   40,    50,    60,    70]
[   54,    68,    82,    96]
[   68,    86,   104,   122]
[   82,   104,   126,   148]

tb.v:110: $finish called at 260 (1s)


Generate the MNIST Test stimulus


Architectural Breakdown: Implementing CNNs via im2col Matrix Multiplication

To deploy an image classification pipeline like MNIST onto a hardware systolic array, the spatial operations of a Convolutional Neural Network (CNN) must be mathematically transformed. While dense layers naturally align with matrix operations, structural convolutions—which involve sliding small weights (kernels) across multi-dimensional input feature maps—do not natively fit a static hardware grid.
To bridge this gap, hardware-accelerated deep learning architectures rely on an operation called im2col (Image to Column). This technique transforms a localized, sliding window convolution into a single, high-throughput General Matrix Multiplication (GEMM) that perfectly maps to a generic systolic array.




1. The im2col Transformation Mechanism

The fundamental concept of im2col is to extract every valid receptive field window from an input image or feature map and serialize it into a distinct column vector.

• Traditional Convolution: A (K x K) kernel slides over an input image, performing an element-wise dot product at every stride location.

• im2col Equivalence: Every (K x K) patch of the input image is unrolled into a linear column vector of length (K^2). If the input has multiple channels (C), the column length becomes (C times K^2).

These extracted columns are stacked side-by-side to form a new, dense input matrix (X_col). Concurrently, the multi-channel convolutional filters are flattened into horizontal row vectors to construct a weight matrix (W_row).

1. Original Input Map & 2x2 Kernel Sliding Window:

   [ 1  2  3 ]

   [ 4  5  6 ] ------> (Extracting 2x2 overlapping patches)

   [ 7  8  9 ]

2. im2col Transformation (Flattening each 2x2 patch into a column vector):

   Patch 1: [1, 2, 4, 5]^T

   Patch 2: [2, 3, 5, 6]^T

   Patch 3: [4, 5, 7, 8]^T
   
   Patch 4: [5, 6, 8, 9]^T

3. Resulting Matrix Structure for Systolic Array GEMM Multiplication:

   W_row (Filters):
   
   [ [-- Filter 0 Weights --] ]
   
   [ [-- Filter 1 Weights --] ]
   

   X_col (Patches):

   [ Patch_1  Patch_2  Patch_3  Patch_4 ]

  e.g. X_col (Patches) =        

                  Column 1    Column 2    Column 3    Column 4
               (Patch 1)   (Patch 2)   (Patch 3)   (Patch 4)
                 ┌   ┐       ┌   ┐       ┌   ┐       ┌   ┐
                 │ 1 │       │ 2 │       │ 4 │       │ 5 │
                 │ 2 │       │ 3 │       │ 5 │       │ 6 │
                 │ 4 │       │ 5 │       │ 7 │       │ 8 │
                 │ 5 │       │ 6 │       │ 8 │       │ 9 │
                 └   ┘       └   ┘       └   ┘       └   ┘

  

By completing this transformation, the spatial sliding-window convolution is simplified into a straightforward matrix multiplication:

Output_gemm = W_row x X_col

The resulting output matrix can then be easily reshaped back into a standard spatial feature map configuration.


2. Application to the Trained MNIST Pipeline

In the implemented MNIST pipeline, the transformation maps a Fully Connected (FC) classification layer, which can be viewed as the limiting case of a convolution where the filter kernel size matches the dimensions of the input image (28 x 28).

  Systolic Array Interface
                                      
[Pixel 0] [Pixel 1] [Pixel 2] ---> (Skewed Columns)
                                               
[Weight 0,0  0,1  0,2]

(Class 0) ---> PE_00 ---- PE_01 ---- PE_02 ...

---> [Accumulator 0]

[Weight 1,0  1,1  1,2]

(Class 1) ---> PE_10 ---- PE_11 ---- PE_12 ...

---> [Accumulator 1]
                                                                        
(Data skews downward and rightward)

1. Dimensional Layout: The 784 pixels of an MNIST image are treated as a single, deep (784 x 1) column vector (X_col). The trained network parameters are arranged as a (10 x 784) matrix ((W_row), where each row represents the weights for one of the 10 target digit classes (0 to 9).

2. Fixed-Point Quantization: Hardware arrays operate efficiently on uniform integers rather than floating-point math. To maintain high accuracy, floating-point weights (W_float in [-1.0, 1.0] from PyTorch are scaled by a factor of 256 (2^8) and clipped into signed 16-bit values (np.int16). Input pixels are similarly converted from ([0.0, 1.0]) to standard ([0, 255]) integer intensities.



3. Data Skewing and Systolic Execution

A standard matrix multiplication calculates a row-by-column dot product all at once. In contrast, a physical Systolic Array passes data step-by-step through a grid of independent Processing Elements (PEs) to minimize data movement and power consumption.

To prevent structural collisions on the shared internal buses, the testbench shifts the input vectors over time:

• Row Skewing (Weights): Row (r) of the weight matrix is delayed by (r) clock cycles before entering the array.

• Column Skewing (Pixels): Column (c) of the image matrix is delayed by (c) clock cycles before entering from the top.

This staggering ensures that the exact weight element (W_{r,k})) arrives at the internal node PE[r][c] at the precise moment the matching pixel value (X_{k,c) passes through.

Every clock cycle, each PE computes a multiply-accumulate step:

C_out <=  C_out + ( A_in x  B_in)

The data then moves across the array. Once the input data stream finishes processing (after 2 x (ROW} - 1) + Length) clock cycles), the accumulated total in each PE holds the final score for that class. The hardware testbench reads these values and applies an Argmax operator to identify the highest score, determining the network's final digit prediction.

In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np

# 1. Download and load real MNIST Test Data
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)

# 2. Define a Single-Layer Neural Network (Matches your 784x10 hardware dimensions)
class MNISTLinearModel(nn.Module):
    def __init__(self):
        super(MNISTLinearModel, self).__init__()
        self.fc = nn.Linear(784, 10, bias=False) # No bias to match simple matrix multiplication

    def forward(self, x):
        x = x.view(-1, 784)
        return self.fc(x)

model = MNISTLinearModel()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 3. Train the model for a few epochs to get true, accurate weights
print("Training model on real MNIST data...")
model.train()
for epoch in range(2): # 2 epochs is enough for ~90% accuracy on this simple layout
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
print("Training complete.")

# 4. Extract weights and quantize them to 16-bit signed fixed-point integers
weights = model.fc.weight.data.numpy() # Shape: (10, 784)

# Scale float weights [-1.0, 1.0] up to fit into integer space safely without overflowing 16 bits
scale_factor = 256
quantized_weights = np.clip(np.round(weights * scale_factor), -32768, 32767).astype(np.int16)

# 5. Extract a subset of real test images and quantize them (0.0-1.0 float scaled to 0-255 integers)
num_test_samples = 10
images_to_save = []
labels_to_save = []

model.eval()
with torch.no_grad():
    for i, (image, label) in enumerate(test_loader):
        if i >= num_test_samples:
            break
        # Flatten image and convert pixels to integers
        img_np = (image.view(784).numpy() * 255).astype(np.int16)
        images_to_save.append(img_np)
        labels_to_save.append(label.item())

# 6. Save everything to disk using Verilog hex format (with explicit int() casting to prevent overflows)
with open('mnist_weights.txt', 'w') as f:
    for row in quantized_weights:
        f.write(" ".join(f"{int(x) & 0xffff:04x}" for x in row) + "\n")

with open('mnist_images.txt', 'w') as f:
    for img in images_to_save:
        f.write(" ".join(f"{int(x) & 0xffff:04x}" for x in img) + "\n")

with open('mnist_labels.txt', 'w') as f:
    for lbl in labels_to_save:
        f.write(f"{lbl}\n")

print(f"\nSuccessfully generated files with REAL MNIST weights and images!")
print(f"Target labels for verification are: {labels_to_save}")


Training model on real MNIST data...
Training complete.

Successfully generated files with REAL MNIST weights and images!
Target labels for verification are: [7, 2, 1, 0, 4, 1, 4, 9, 5, 9]


In [26]:
%%writefile sysarray.v
module SystolicArrayMod #(
    parameter DATA_WIDTH = 16,
    parameter ACC_WIDTH  = 32,
    parameter ROW        = 10,  // Dynamic matching for classification sizes
    parameter COL        = 1
)(
    input wire Clk, Rst,
    // Flattened input/output bit vectors for absolute parametric scaling support
    input wire signed [(DATA_WIDTH*ROW)-1:0] A_rows_flat,
    input wire signed [(DATA_WIDTH*COL)-1:0] B_cols_flat,
    output wire signed [(ACC_WIDTH*ROW*COL)-1:0] C_flat
);

    wire signed [DATA_WIDTH-1:0] A [0:ROW-1][0:COL];
    wire signed [DATA_WIDTH-1:0] B [0:ROW][0:COL-1];
    wire signed [ACC_WIDTH-1:0]  C [0:ROW-1][0:COL-1];

    // Unflatten individual wire inputs mathematically using generate blocks
    genvar i, j;
    generate
        for(i = 0; i < ROW; i = i + 1) begin: unpack_a
            assign A[i][0] = A_rows_flat[(i*DATA_WIDTH) +: DATA_WIDTH];
        end
        for(j = 0; j < COL; j = j + 1) begin: unpack_b
            assign B[0][j] = B_cols_flat[(j*DATA_WIDTH) +: DATA_WIDTH];
        end
        for(i = 0; i < ROW; i = i + 1) begin: pack_c
            for(j = 0; j < COL; j = j + 1) begin: pack_c_inner
                assign C_flat[((i*COL + j)*ACC_WIDTH) +: ACC_WIDTH] = C[i][j];
            end
        end
    endgenerate

    // Structural Array Generation remains perfectly generic via GenVar blocks
    genvar r, c;
    generate
        for (r=0; r<ROW; r=r+1) begin: genrow
            for (c=0; c<COL; c=c+1) begin: gencol
                PE PE_inst (
                    .clk   (Clk),
                    .rst   (Rst),
                    .a_in  (A[r][c]),
                    .b_in  (B[r][c]),
                    .a_out (A[r][c+1]),
                    .b_out (B[r+1][c]),
                    .c_out (C[r][c])
                );
            end
        end
    endgenerate

endmodule


Overwriting sysarray.v


In [27]:
%%writefile tb.v
module testbench;

    // Architectural adjustments to map a Fully Connected Neural Network Layer
    parameter DATA_WIDTH   = 16;
    parameter ACC_WIDTH    = 32;
    parameter ROW          = 10;   // 10 output classes (Digits 0-9)
    parameter COL          = 1;    // Processing a single image frame column loop at a time
    parameter MNIST_LENGTH = 784;  // Length of a flattened 28x28 input vector
    parameter TEST_IMAGES  = 10;   // Ingestion sample test depth

    reg clk_test;
    reg rst_test;

    // Interconnect registers to feed the flattened system array
    reg signed [(DATA_WIDTH*ROW)-1:0] A_rows_flat;
    reg signed [(DATA_WIDTH*COL)-1:0] B_cols_flat;
    wire signed [(ACC_WIDTH*ROW*COL)-1:0] C_flat;

    // Memories for loading MNIST file datasets
    reg signed [DATA_WIDTH-1:0] image_mem [0:TEST_IMAGES-1][0:MNIST_LENGTH-1];
    reg signed [DATA_WIDTH-1:0] weight_mem [0:ROW-1][0:MNIST_LENGTH-1];
    reg [3:0] label_mem [0:TEST_IMAGES-1];

    // Unpacked internal monitoring buffers
    reg signed [ACC_WIDTH-1:0] final_classifications [0:ROW-1];

    // Iteration indexes
    integer img_idx, t, r, c;
    integer col_idx, row_idx;
    integer correct_predictions;

    // Argmax evaluation nodes
    reg signed [ACC_WIDTH-1:0] max_val;
    integer predicted_digit;

    // Instantiate your updated, scalable generic Systolic Array configuration
    SystolicArrayMod #(
        .DATA_WIDTH(DATA_WIDTH),
        .ACC_WIDTH(ACC_WIDTH),
        .ROW(ROW),
        .COL(COL)
    ) dut (
        .Clk(clk_test),
        .Rst(rst_test),
        .A_rows_flat(A_rows_flat),
        .B_cols_flat(B_cols_flat),
        .C_flat(C_flat)
    );

    // Continuous assignment to extract the output matrix calculation array back into readable parts
    genvar k;
    generate
        for(k=0; k<ROW; k=k+1) begin: extract_out
            always @(*) begin
                final_classifications[k] = C_flat[(k*ACC_WIDTH) +: ACC_WIDTH];
            end
        end
    endgenerate

    // Clock Engine Definition (50MHz)
    always #10 clk_test = ~clk_test;

    initial begin
        // 1. Data File Parse Sequence
        $readmemh("mnist_images.txt", image_mem);
        $readmemh("mnist_weights.txt", weight_mem);
        $readmemh("mnist_labels.txt", label_mem);

        clk_test = 0;
        correct_predictions = 0;
        A_rows_flat = 0;
        B_cols_flat = 0;

        $display("\n========================================================");
        $display("   STARTING MNIST SYSTEM-ARRAY CNN INFERENCE PIPELINE  ");
        $display("========================================================\n");

        // 2. Loop Through Each Image in the Dataset
        for (img_idx = 0; img_idx < TEST_IMAGES; img_idx = img_idx + 1) begin

            // Hard reset the accumulation memory cells inside the PEs between inferences
            rst_test = 1;
            #20;
            rst_test = 0;

            // 3. Symmetric Skew Processing Loop for 784 Vector Elements
            for (t = 0; t < ((2 * (ROW - 1)) + MNIST_LENGTH); t = t + 1) begin
                @(negedge clk_test);

                // Map CNN Pre-Trained Weights into A row data ports with skewed phase positioning
                for (r = 0; r < ROW; r = r + 1) begin
                    col_idx = t - r;
                    if (col_idx >= 0 && col_idx < MNIST_LENGTH) begin
                        A_rows_flat[(r*DATA_WIDTH) +: DATA_WIDTH] = weight_mem[r][col_idx];
                    end else begin
                        A_rows_flat[(r*DATA_WIDTH) +: DATA_WIDTH] = {DATA_WIDTH{1'b0}};
                    end
                end

                // Map Target Image Active Pixels into B column data ports with skewed phase positioning
                for (c = 0; c < COL; c = c + 1) begin
                    row_idx = t - c;
                    if (row_idx >= 0 && row_idx < MNIST_LENGTH) begin
                        B_cols_flat[(c*DATA_WIDTH) +: DATA_WIDTH] = image_mem[img_idx][row_idx];
                    end else begin
                        B_cols_flat[(c*DATA_WIDTH) +: DATA_WIDTH] = {DATA_WIDTH{1'b0}};
                    end
                end
            end

            // Wait a final clock edge cycle for calculation results to settle
            @(negedge clk_test);

            // 4. Implement Argmax Verification Core Logic
            max_val = final_classifications[0];
            predicted_digit = 0;

            for (r = 1; r < ROW; r = r + 1) begin
                if (final_classifications[r] > max_val) begin
                    max_val = final_classifications[r];
                    predicted_digit = r;
                end
            end

            // Compare classification score against true source target label
            if (predicted_digit == label_mem[img_idx]) begin
                correct_predictions = correct_predictions + 1;
                $display("Image [%0d] -> Predicted: %0d | True Label: %0d -> [ MATCH ]", img_idx, predicted_digit, label_mem[img_idx]);
            end else begin
                $display("Image [%0d] -> Predicted: %0d | True Label: %0d -> [ MISMATCH ]", img_idx, predicted_digit, label_mem[img_idx]);
            end
        end

        // 5. Final Suite Status Output
        $display("\n========================================================");
        $display(" MNIST PIPELINE VERIFICATION COMPLETE");
        $display(" Successfully Verified: %0d / %0d Images", correct_predictions, TEST_IMAGES);
        $display(" Final Accuracy Score:  %0d%%", (correct_predictions * 100) / TEST_IMAGES);
        $display("========================================================\n");
        $finish;
    end

endmodule


Overwriting tb.v


In [28]:
!iverilog -o mnist_sim PE.v sysarray.v tb.v
!vvp mnist_sim



   STARTING MNIST SYSTEM-ARRAY CNN INFERENCE PIPELINE  

Image [0] -> Predicted: 7 | True Label: 7 -> [ MATCH ]
Image [1] -> Predicted: 2 | True Label: 2 -> [ MATCH ]
Image [2] -> Predicted: 1 | True Label: 1 -> [ MATCH ]
Image [3] -> Predicted: 0 | True Label: 0 -> [ MATCH ]
Image [4] -> Predicted: 4 | True Label: 4 -> [ MATCH ]
Image [5] -> Predicted: 1 | True Label: 1 -> [ MATCH ]
Image [6] -> Predicted: 4 | True Label: 4 -> [ MATCH ]
Image [7] -> Predicted: 9 | True Label: 9 -> [ MATCH ]
Image [8] -> Predicted: 6 | True Label: 5 -> [ MISMATCH ]
Image [9] -> Predicted: 9 | True Label: 9 -> [ MATCH ]

 MNIST PIPELINE VERIFICATION COMPLETE
 Successfully Verified: 9 / 10 Images
 Final Accuracy Score:  90%

tb.v:140: $finish called at 160600 (1s)
